# 🤖 Agentic RAG — arXiv Research Assistant
### Step-by-step walkthrough notebook

---

This notebook walks through every component of the Agentic RAG system **in order**, 
from environment setup all the way to a full agent run. Each cell is self-contained 
and explained so you can see what happens at every stage.

**Pipeline overview:**
```
Step 1 → Install packages
Step 2 → config.py          — load settings from .env
Step 3 → arxiv_fetcher.py   — search & download papers
Step 4 → document_processor.py — extract text & chunk
Step 5 → vector_store.py    — embed chunks into FAISS
Step 6 → rag_chain.py       — single-turn RAG Q&A
Step 7 → rag_chain.py       — conversational RAG (multi-turn)
Step 8 → research_agent.py  — full autonomous agent
Step 9 → end-to-end demo
```

**Before running:** Make sure your `.env` file exists in the same folder as this notebook with `ANTHROPIC_API_KEY=sk-ant-...`

---
## Step 1 — Install all required packages

This installs every library the project needs. Run this once.
If you already ran `pip install -r requirements.txt`, you can skip this cell.

In [ ]:
# Install all dependencies
# This may take 3-5 minutes on first run

!pip install -q \
    langchain langchain-community langchain-anthropic langchain-openai langchain-text-splitters \
    langchain-huggingface \
    faiss-cpu>=1.9.0.post1 \
    arxiv pypdf pymupdf \
    sentence-transformers tiktoken \
    python-dotenv pydantic tenacity loguru \
    streamlit

print("✅ All packages installed")

In [ ]:
# Verify the key imports work correctly

import langchain
import arxiv
import faiss
import sentence_transformers
from dotenv import load_dotenv
import os

print(f"✅ langchain          version: {langchain.__version__}")
print(f"✅ arxiv              version: {arxiv.__version__}")
print(f"✅ faiss              version: {faiss.__version__}")
print(f"✅ sentence_transformers       imported")
print()
print("All core packages are available. Ready to proceed!")

---
## Step 2 — `config.py` — Load settings from `.env`

**What this does:**  
`config.py` reads your `.env` file using `python-dotenv` and exposes every 
setting as a typed Python attribute via Pydantic. This is the single source 
of truth for all configuration — every other module imports `settings` from here.

**Why it matters:**  
- Your API key never appears in source code
- All settings have sensible defaults (app works even with a minimal `.env`)
- Switching from Claude → Ollama is just changing one `.env` line

In [ ]:
# First, check if .env exists and has the API key
from pathlib import Path

env_path = Path(".env")
if env_path.exists():
    print("✅ .env file found")
    # Show keys present (without showing actual values)
    lines = env_path.read_text().strip().splitlines()
    keys = [l.split("=")[0] for l in lines if l and not l.startswith("#")]
    print(f"   Keys configured: {keys}")
else:
    print("❌ .env file NOT found")
    print("   Create a .env file with at minimum:")
    print("   PROVIDER=anthropic")
    print("   ANTHROPIC_API_KEY=sk-ant-...")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# config.py — full implementation
# ─────────────────────────────────────────────────────────────────
# This is the exact contents of config.py from the project.
# Running this cell defines the Settings class and creates
# the global `settings` object used everywhere else.

import os
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()  # reads .env → loads into os.environ

class Settings(BaseModel):
    # ── LLM Provider ──────────────────────────────────────────────
    PROVIDER: str = os.getenv("PROVIDER", "anthropic")

    OPENAI_API_KEY: str  = os.getenv("OPENAI_API_KEY", "")
    OPENAI_MODEL:   str  = os.getenv("OPENAI_MODEL", "gpt-4o")

    ANTHROPIC_API_KEY: str = os.getenv("ANTHROPIC_API_KEY", "")
    ANTHROPIC_MODEL:   str = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")

    # ── Local LLM options ─────────────────────────────────────────
    OLLAMA_BASE_URL: str = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.getenv("OLLAMA_MODEL", "llama3.1")

    HF_MODEL_ID:       str = os.getenv("HF_MODEL_ID", "microsoft/Phi-3.5-mini-instruct")
    HF_DEVICE:         str = os.getenv("HF_DEVICE", "cpu")
    HF_MAX_NEW_TOKENS: int = int(os.getenv("HF_MAX_NEW_TOKENS", "512"))

    LLAMACPP_MODEL_PATH:   str = os.getenv("LLAMACPP_MODEL_PATH", "./models/model.gguf")
    LLAMACPP_N_CTX:        int = int(os.getenv("LLAMACPP_N_CTX", "4096"))
    LLAMACPP_N_GPU_LAYERS: int = int(os.getenv("LLAMACPP_N_GPU_LAYERS", "0"))

    # ── Embeddings ────────────────────────────────────────────────
    EMBEDDING_MODEL: str = os.getenv(
        "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
    )

    # ── Vector Store ──────────────────────────────────────────────
    VECTOR_STORE_TYPE: str = os.getenv("VECTOR_STORE_TYPE", "faiss")
    VECTOR_STORE_PATH: str = os.getenv("VECTOR_STORE_PATH", "./data/vector_store")
    CHROMA_COLLECTION: str = os.getenv("CHROMA_COLLECTION", "arxiv_papers")

    # ── arXiv ─────────────────────────────────────────────────────
    ARXIV_MAX_RESULTS: int = int(os.getenv("ARXIV_MAX_RESULTS", "10"))
    ARXIV_DOWNLOAD_DIR: str = os.getenv("ARXIV_DOWNLOAD_DIR", "./data/papers")
    ARXIV_SORT_BY: str = os.getenv("ARXIV_SORT_BY", "relevance")

    # ── Text Splitting ────────────────────────────────────────────
    CHUNK_SIZE:    int = int(os.getenv("CHUNK_SIZE", "1000"))
    CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", "200"))

    # ── Retrieval ─────────────────────────────────────────────────
    TOP_K_DOCS:     int = int(os.getenv("TOP_K_DOCS", "5"))
    RETRIEVAL_TYPE: str = os.getenv("RETRIEVAL_TYPE", "mmr")

    # ── Agent ─────────────────────────────────────────────────────
    AGENT_MAX_ITERATIONS: int  = int(os.getenv("AGENT_MAX_ITERATIONS", "10"))
    AGENT_VERBOSE:        bool = os.getenv("AGENT_VERBOSE", "true").lower() == "true"

settings = Settings()

# ── Inspect what was loaded ───────────────────────────────────────
print("=" * 50)
print("Settings loaded from .env:")
print("=" * 50)
print(f"  PROVIDER:          {settings.PROVIDER}")
print(f"  ANTHROPIC_MODEL:   {settings.ANTHROPIC_MODEL}")
print(f"  ANTHROPIC_API_KEY: {'✅ Set' if settings.ANTHROPIC_API_KEY else '❌ Missing'}")
print(f"  EMBEDDING_MODEL:   {settings.EMBEDDING_MODEL}")
print(f"  VECTOR_STORE_TYPE: {settings.VECTOR_STORE_TYPE}")
print(f"  CHUNK_SIZE:        {settings.CHUNK_SIZE}  ← int, not string")
print(f"  CHUNK_OVERLAP:     {settings.CHUNK_OVERLAP}")
print(f"  TOP_K_DOCS:        {settings.TOP_K_DOCS}")
print(f"  RETRIEVAL_TYPE:    {settings.RETRIEVAL_TYPE}")
print(f"  AGENT_VERBOSE:     {settings.AGENT_VERBOSE}  ← bool, not string")
print(f"  ARXIV_MAX_RESULTS: {settings.ARXIV_MAX_RESULTS}")
print("=" * 50)

---
## Step 3 — `arxiv_fetcher.py` — Search & Download Papers

**What this does:**  
Wraps the `arxiv` Python package to search arXiv, return rich metadata, 
and download PDFs to a local folder.

**Two-stage process:**
1. `search()` — hits the arXiv API, returns metadata only (fast, no download)
2. `download_pdfs()` — downloads the actual PDF files (slower, rate-limited)

**Key design choices:**
- `tenacity` retry decorator handles transient arXiv API failures automatically
- 2-second delay between PDF downloads to respect arXiv's rate limits
- Already-downloaded PDFs are skipped (idempotent — safe to re-run)
- Returns a clean `ArxivPaper` dataclass instead of raw arxiv objects

In [ ]:
# ─────────────────────────────────────────────────────────────────
# arxiv_fetcher.py — Part 1: the ArxivPaper dataclass
# ─────────────────────────────────────────────────────────────────
# Every paper returned by the fetcher is wrapped in this dataclass.
# It holds all metadata plus the optional local PDF path.

from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ArxivPaper:
    """A single arXiv paper with metadata and optional local PDF path."""
    paper_id:       str
    title:          str
    authors:        List[str]
    abstract:       str
    published:      str
    updated:        str
    url:            str
    pdf_url:        str
    categories:     List[str]
    local_pdf_path: Optional[str] = None
    _raw:           Optional[object] = field(default=None, repr=False)

    def to_dict(self) -> dict:
        return {
            "paper_id":       self.paper_id,
            "title":          self.title,
            "authors":        self.authors,
            "abstract":       self.abstract,
            "published":      self.published,
            "url":            self.url,
            "categories":     self.categories,
            "local_pdf_path": self.local_pdf_path,
        }

print("✅ ArxivPaper dataclass defined")
print()
print("Fields:")
for f in ArxivPaper.__dataclass_fields__:
    print(f"  {f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# arxiv_fetcher.py — Part 2: the ArxivFetcher class
# ─────────────────────────────────────────────────────────────────

import time
from pathlib import Path
import arxiv
from tenacity import retry, stop_after_attempt, wait_exponential

class ArxivFetcher:
    """
    Searches arXiv and downloads PDFs.
    Stage 1: search()        → returns metadata (fast)
    Stage 2: download_pdfs() → saves PDFs locally (slower)
    """

    SORT_CRITERIA = {
        "relevance":       arxiv.SortCriterion.Relevance,
        "lastUpdatedDate": arxiv.SortCriterion.LastUpdatedDate,
        "submittedDate":   arxiv.SortCriterion.SubmittedDate,
    }

    def __init__(self, download_dir=None, max_results=None, sort_by=None):
        self.download_dir   = Path(download_dir or settings.ARXIV_DOWNLOAD_DIR)
        self.max_results    = max_results or settings.ARXIV_MAX_RESULTS
        self.sort_criterion = self.SORT_CRITERIA.get(
            sort_by or settings.ARXIV_SORT_BY,
            arxiv.SortCriterion.Relevance
        )
        self.download_dir.mkdir(parents=True, exist_ok=True)
        # Polite client: 3s delay, 5 retries
        self.client = arxiv.Client(page_size=50, delay_seconds=3, num_retries=5)

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=10))
    def search(self, query: str, max_results: int = None,
               categories: List[str] = None) -> List[ArxivPaper]:
        """Search arXiv. Returns metadata only — no PDFs downloaded yet."""
        n = max_results or self.max_results

        # Optionally narrow to specific arXiv categories
        if categories:
            cat_filter = " OR ".join(f"cat:{c}" for c in categories)
            full_query = f"({query}) AND ({cat_filter})"
        else:
            full_query = query

        print(f"🔍 Searching arXiv: '{full_query}' (max {n} results)...")

        search_obj = arxiv.Search(
            query=full_query,
            max_results=n,
            sort_by=self.sort_criterion,
        )

        papers = []
        for result in self.client.results(search_obj):
            paper = ArxivPaper(
                paper_id=result.entry_id.split("/")[-1],
                title=result.title,
                authors=[a.name for a in result.authors],
                abstract=result.summary,
                published=result.published.strftime("%Y-%m-%d"),
                updated=result.updated.strftime("%Y-%m-%d"),
                url=result.entry_id,
                pdf_url=result.pdf_url,
                categories=result.categories,
                _raw=result,
            )
            papers.append(paper)

        print(f"✅ Found {len(papers)} papers")
        return papers

    def download_pdfs(self, papers: List[ArxivPaper],
                      delay: float = 2.0) -> List[ArxivPaper]:
        """Download PDFs for a list of papers. Skips already-downloaded files."""
        for i, paper in enumerate(papers):
            pdf_path = self.download_dir / f"{paper.paper_id}.pdf"
            if pdf_path.exists():
                print(f"  [{i+1}/{len(papers)}] Already downloaded: {pdf_path.name}")
                paper.local_pdf_path = str(pdf_path)
                continue
            try:
                print(f"  [{i+1}/{len(papers)}] Downloading: {paper.title[:55]}...")
                if paper._raw:
                    paper._raw.download_pdf(
                        dirpath=str(self.download_dir),
                        filename=f"{paper.paper_id}.pdf"
                    )
                paper.local_pdf_path = str(pdf_path)
                print(f"     ✅ Saved → {pdf_path.name}")
                if i < len(papers) - 1:
                    time.sleep(delay)
            except Exception as e:
                print(f"     ❌ Failed: {e}")
        return papers

    def fetch_by_ids(self, paper_ids: List[str]) -> List[ArxivPaper]:
        """Fetch specific papers by arXiv ID (e.g. ['2210.03629'])."""
        search_obj = arxiv.Search(id_list=paper_ids)
        papers = []
        for result in self.client.results(search_obj):
            papers.append(ArxivPaper(
                paper_id=result.entry_id.split("/")[-1],
                title=result.title,
                authors=[a.name for a in result.authors],
                abstract=result.summary,
                published=result.published.strftime("%Y-%m-%d"),
                updated=result.updated.strftime("%Y-%m-%d"),
                url=result.entry_id,
                pdf_url=result.pdf_url,
                categories=result.categories,
                _raw=result,
            ))
        return papers

print("✅ ArxivFetcher class defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Search arXiv for papers on Agentic AI
# ─────────────────────────────────────────────────────────────────
# This actually hits the arXiv API and returns real papers.
# We search for 3 papers to keep things fast for demo purposes.

fetcher = ArxivFetcher(max_results=3)
papers  = fetcher.search("agentic AI ReAct autonomous agents LLM")

print()
print("=" * 60)
print(f"Papers found: {len(papers)}")
print("=" * 60)

for i, p in enumerate(papers, 1):
    print(f"\n[{i}] {p.title}")
    print(f"    ID:        {p.paper_id}")
    print(f"    Authors:   {', '.join(p.authors[:3])}{'...' if len(p.authors) > 3 else ''}")
    print(f"    Published: {p.published}")
    print(f"    URL:       {p.url}")
    print(f"    Categories:{p.categories[:3]}")
    print(f"    Abstract:  {p.abstract[:200]}...")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Download PDFs
# ─────────────────────────────────────────────────────────────────
# Downloads PDFs to ./data/papers/<paper_id>.pdf
# Re-running this cell is safe — already-downloaded files are skipped.
# Expect this to take ~10-20 seconds for 3 papers.

print("Downloading PDFs...")
print(f"Destination: {fetcher.download_dir.resolve()}")
print()

papers = fetcher.download_pdfs(papers, delay=2.0)

print()
print("=" * 60)
print("Download summary:")
for p in papers:
    status = "✅" if p.local_pdf_path and Path(p.local_pdf_path).exists() else "❌"
    size   = Path(p.local_pdf_path).stat().st_size // 1024 if p.local_pdf_path and Path(p.local_pdf_path).exists() else 0
    print(f"  {status} {p.paper_id}.pdf  ({size} KB)")

downloaded = [p for p in papers if p.local_pdf_path]
print(f"\n{len(downloaded)}/{len(papers)} papers downloaded successfully")

---
## Step 4 — `document_processor.py` — Extract Text & Chunk

**What this does:**  
Reads each PDF (using PyMuPDF → PyPDF fallback), cleans the extracted text,  
then splits it into overlapping chunks ready for embedding.

**Why chunking matters:**  
LLMs have a context window limit (~200K tokens for Claude, but embeddings need 
much smaller chunks ~1000 chars). We split each paper into small chunks so we  
can retrieve only the *relevant* pieces instead of feeding the entire paper.

**Overlap explained:**  
With `chunk_size=1000` and `chunk_overlap=200`, the last 200 chars of chunk N  
appear again at the start of chunk N+1. This prevents a sentence that spans  
a boundary from losing context on either side.

```
Chunk 0: [-------- 1000 chars --------]
Chunk 1:           [---- 200 overlap -------- 1000 chars --------]
Chunk 2:                              [---- 200 overlap -------- 1000 chars --------]
```

In [ ]:
# ─────────────────────────────────────────────────────────────────
# document_processor.py — full implementation
# ─────────────────────────────────────────────────────────────────

import re
from typing import List, Optional
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

class DocumentProcessor:
    """
    Converts ArxivPaper objects (with local PDFs) into chunked
    LangChain Documents ready for embedding.
    """

    def __init__(self, chunk_size=None, chunk_overlap=None):
        cs = chunk_size    or settings.CHUNK_SIZE
        co = chunk_overlap or settings.CHUNK_OVERLAP
        # Tries separators in order: paragraph → line → sentence → word → char
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=cs,
            chunk_overlap=co,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )
        print(f"DocumentProcessor ready (chunk_size={cs}, overlap={co})")

    def process_papers(self, papers: List[ArxivPaper]) -> List[Document]:
        """Process a list of ArxivPaper objects → flat list of Document chunks."""
        all_docs = []
        for paper in papers:
            try:
                if paper.local_pdf_path and Path(paper.local_pdf_path).exists():
                    docs = self._process_pdf(paper)
                    source = "PDF"
                else:
                    print(f"  ⚠ No PDF for {paper.paper_id} — using abstract")
                    docs = self._process_abstract(paper)
                    source = "abstract"
                all_docs.extend(docs)
                print(f"  ✅ [{paper.paper_id}] → {len(docs)} chunks ({source})")
            except Exception as e:
                print(f"  ❌ Failed to process {paper.paper_id}: {e}")
                all_docs.extend(self._process_abstract(paper))
        print(f"\nTotal chunks produced: {len(all_docs)}")
        return all_docs

    def _process_pdf(self, paper):
        text = self._extract_text_pymupdf(paper.local_pdf_path)
        if not text or len(text.strip()) < 200:
            text = self._extract_text_pypdf(paper.local_pdf_path)
        return self._chunk(self._clean_text(text), paper)

    def _extract_text_pymupdf(self, pdf_path: str) -> str:
        try:
            import fitz  # PyMuPDF
            doc   = fitz.open(pdf_path)
            pages = [page.get_text("text") for page in doc]
            doc.close()
            return "\n\n".join(pages)
        except Exception as e:
            print(f"    PyMuPDF error: {e}")
            return ""

    def _extract_text_pypdf(self, pdf_path: str) -> str:
        try:
            from pypdf import PdfReader
            reader = PdfReader(pdf_path)
            return "\n\n".join(page.extract_text() or "" for page in reader.pages)
        except Exception as e:
            print(f"    PyPDF error: {e}")
            return ""

    def _process_abstract(self, paper):
        text = f"Title: {paper.title}\n\nAbstract: {paper.abstract}"
        return self._chunk(text, paper)

    def _chunk(self, text: str, paper) -> List[Document]:
        if not text.strip():
            return []
        meta = {
            "paper_id":  paper.paper_id,
            "title":     paper.title,
            "authors":   ", ".join(paper.authors[:3]) + (" et al." if len(paper.authors) > 3 else ""),
            "published": paper.published,
            "url":       paper.url,
            "pdf_url":   paper.pdf_url,
            "categories":paper.categories[0] if paper.categories else "",
            "source":    "arxiv",
        }
        chunks = self.splitter.split_text(text)
        return [
            Document(
                page_content=chunk,
                metadata={**meta, "chunk_index": i, "total_chunks": len(chunks)}
            )
            for i, chunk in enumerate(chunks)
        ]

    @staticmethod
    def _clean_text(text: str) -> str:
        if not text:
            return ""
        text = text.replace("\f", "\n").replace("\r", "\n")
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r" {2,}", " ", text)
        lines = [l for l in text.split("\n") if not re.match(r"^\s*\d{1,3}\s*$", l)]
        return "\n".join(lines).strip()

print("✅ DocumentProcessor class defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Process the downloaded papers into chunks
# ─────────────────────────────────────────────────────────────────

processor = DocumentProcessor()
print()
print("Processing papers...")
print()
docs = processor.process_papers(papers)

# ── Inspect a sample chunk ────────────────────────────────────────
print()
print("=" * 60)
print("Sample chunk (doc[5]):")
print("=" * 60)
sample = docs[5] if len(docs) > 5 else docs[0]

print("METADATA:")
for k, v in sample.metadata.items():
    print(f"  {k}: {v}")

print()
print(f"CONTENT ({len(sample.page_content)} chars):")
print(sample.page_content[:500])
print("...")

# ── Chunk size distribution ───────────────────────────────────────
sizes = [len(d.page_content) for d in docs]
print()
print("Chunk size stats:")
print(f"  Min:  {min(sizes)} chars")
print(f"  Max:  {max(sizes)} chars")
print(f"  Avg:  {sum(sizes)//len(sizes)} chars")
print(f"  Total:{len(docs)} chunks across {len(papers)} papers")

---
## Step 5 — `vector_store.py` — Embed & Index with FAISS

**What this does:**  
Takes the document chunks, converts each one to a dense vector using a 
sentence-transformer model, then builds a FAISS index for fast similarity search.

**Key concepts:**
- **Embedding:** Each chunk of text → a 384-dimensional float vector that 
  captures its semantic meaning
- **FAISS:** Facebook AI Similarity Search — an index that can find the 
  nearest vectors to a query vector extremely fast, even with millions of vectors
- **MMR (Maximal Marginal Relevance):** Retrieves top-K results that are 
  *relevant* to the query AND *diverse* from each other (avoids returning 
  5 chunks all saying the same thing)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# vector_store.py — full implementation
# ─────────────────────────────────────────────────────────────────

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import VectorStore

def get_embeddings():
    """
    Returns the embedding model.
    Default: HuggingFace all-MiniLM-L6-v2 (local, free, ~90MB download once)
    Alternative: OpenAI text-embedding-3-small (needs API key)
    """
    model_name = settings.EMBEDDING_MODEL

    if model_name.startswith("text-embedding") and settings.OPENAI_API_KEY:
        from langchain_openai import OpenAIEmbeddings
        print(f"Using OpenAI embeddings: {model_name}")
        return OpenAIEmbeddings(model=model_name, openai_api_key=settings.OPENAI_API_KEY)

    print(f"Using HuggingFace embeddings: {model_name}")
    print("(First run downloads ~90MB model — subsequent runs use cache)")
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


class VectorStoreManager:
    """Wraps FAISS or ChromaDB with a consistent interface."""

    def __init__(self, store_type=None, persist_path=None):
        self.store_type   = (store_type   or settings.VECTOR_STORE_TYPE).lower()
        self.persist_path = Path(persist_path or settings.VECTOR_STORE_PATH)
        self.persist_path.mkdir(parents=True, exist_ok=True)
        self.embeddings   = get_embeddings()
        self._store       = None

    def load_or_create(self, documents=None) -> VectorStore:
        """
        Smart loader:
        - No existing store + documents → create new store
        - Existing store + no documents → load from disk
        - Existing store + documents    → load and append
        """
        existing = self._try_load()
        if existing and documents:
            print("Loading existing store + adding new documents...")
            existing.add_documents(documents)
            self._save(existing)
            self._store = existing
        elif existing:
            print("Loaded existing store from disk.")
            self._store = existing
        elif documents:
            print(f"Creating new {self.store_type} vector store...")
            store = self._create(documents)
            self._save(store)
            self._store = store
        else:
            raise ValueError("No store found and no documents provided.")
        return self._store

    def add_documents(self, documents):
        if self._store is None:
            self.load_or_create(documents)
            return
        self._store.add_documents(documents)
        self._save(self._store)

    def similarity_search(self, query: str, k=None):
        self._require_store()
        return self._store.similarity_search(query, k=k or settings.TOP_K_DOCS)

    def as_retriever(self, search_type=None, k=None):
        """
        Returns a LangChain-compatible retriever.
        search_type='mmr'        → diverse results (recommended)
        search_type='similarity' → pure similarity
        """
        self._require_store()
        st = search_type or settings.RETRIEVAL_TYPE
        kk = k          or settings.TOP_K_DOCS
        kwargs = {"k": kk}
        if st == "mmr":
            kwargs["fetch_k"] = kk * 3  # fetch 3x, pick most diverse k
        return self._store.as_retriever(search_type=st, search_kwargs=kwargs)

    def _try_load(self):
        try:
            if self.store_type == "faiss":
                return self._load_faiss()
        except Exception:
            pass
        return None

    def _create(self, documents):
        if self.store_type == "faiss":
            return self._create_faiss(documents)
        raise ValueError(f"Unknown store type: {self.store_type}")

    def _save(self, store):
        if self.store_type == "faiss":
            store.save_local(str(self.persist_path))
            print(f"  Saved FAISS index → {self.persist_path}")

    def _create_faiss(self, documents):
        from langchain_community.vectorstores import FAISS
        return FAISS.from_documents(documents, self.embeddings)

    def _load_faiss(self):
        from langchain_community.vectorstores import FAISS
        idx = self.persist_path / "index.faiss"
        if not idx.exists():
            raise FileNotFoundError(f"No FAISS index at {idx}")
        return FAISS.load_local(
            str(self.persist_path), self.embeddings,
            allow_dangerous_deserialization=True
        )

    def _require_store(self):
        if self._store is None:
            raise RuntimeError("Call load_or_create() first.")

print("✅ VectorStoreManager class defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Embed chunks and build FAISS index
# ─────────────────────────────────────────────────────────────────
# This embeds all chunks into vectors and saves the FAISS index.
# First run downloads the embedding model (~90MB).
# Subsequent runs load from disk instantly.

print("Building vector store...")
print()
vsm = VectorStoreManager()
print()
store = vsm.load_or_create(documents=docs)

print()
print("=" * 60)
print("Vector store ready!")

# ── Test similarity search ────────────────────────────────────────
print()
print("Test query: 'how does planning work in autonomous agents?'")
results = vsm.similarity_search("how does planning work in autonomous agents?", k=3)
print(f"Retrieved {len(results)} chunks:")
for i, r in enumerate(results, 1):
    print(f"\n  [{i}] {r.metadata['title'][:55]}...")
    print(f"      chunk {r.metadata['chunk_index']}/{r.metadata['total_chunks']}")
    print(f"      {r.page_content[:150]}...")

---
## Step 6 — `rag_chain.py` — Single-turn RAG Pipeline

**What this does:**  
Builds a LangChain LCEL pipeline that chains retrieval + prompt filling + LLM 
into a single callable. This is the core RAG implementation.

**The pipeline in plain English:**
1. User question comes in
2. Retriever fetches the 5 most relevant chunks from FAISS
3. Chunks are formatted into a numbered context block
4. Prompt is filled: system instructions + context + question
5. Claude reads the filled prompt and generates a cited answer
6. Answer is returned as a plain string

**LCEL pipe syntax:**  
`chain = { "context": retriever | formatter, "question": passthrough } | prompt | llm | parser`  
Each `|` passes its output as the input to the next component.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# rag_chain.py — Part 1: LLM factory
# ─────────────────────────────────────────────────────────────────
# get_llm() returns whichever LLM is configured in .env
# All other code uses this function — never imports LLM classes directly.

def get_llm(streaming: bool = False):
    provider = settings.PROVIDER.lower()

    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        print(f"Using Anthropic: {settings.ANTHROPIC_MODEL}")
        return ChatAnthropic(
            model=settings.ANTHROPIC_MODEL,
            anthropic_api_key=settings.ANTHROPIC_API_KEY,
            streaming=streaming,
            temperature=0.2,
        )

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        print(f"Using OpenAI: {settings.OPENAI_MODEL}")
        return ChatOpenAI(
            model=settings.OPENAI_MODEL,
            openai_api_key=settings.OPENAI_API_KEY,
            streaming=streaming,
            temperature=0.2,
        )

    if provider == "ollama":
        from langchain_ollama import ChatOllama
        print(f"Using Ollama: {settings.OLLAMA_MODEL}")
        return ChatOllama(
            model=settings.OLLAMA_MODEL,
            base_url=settings.OLLAMA_BASE_URL,
            temperature=0.2,
        )

    raise ValueError(f"Unknown provider: {settings.PROVIDER}")


# ── Test: instantiate the LLM ─────────────────────────────────────
llm = get_llm()
print(f"✅ LLM ready: {type(llm).__name__}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# rag_chain.py — Part 2: the RAGChain class
# ─────────────────────────────────────────────────────────────────

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# The system prompt — this is what makes Claude behave as a research assistant
SYSTEM_PROMPT = """\
You are an expert research assistant specializing in Agentic AI and machine learning.
Your role is to help researchers navigate complex arXiv papers efficiently.

You have been given relevant excerpts from research papers (the "context").
Use ONLY this context to answer. If the answer is not in the context, say so clearly.

Guidelines:
- Cite paper titles and authors when referencing specific claims.
- Highlight key contributions, methods, and findings.
- Explain technical jargon in plain language when helpful.
- Be concise but thorough.

Context from retrieved papers:
{context}
"""

def _format_docs(docs):
    """Format retrieved chunks into a numbered context block."""
    parts = []
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        header = f"[{i}] {m.get('title','?')} ({m.get('authors','')}, {m.get('published','')})"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


class RAGChain:
    """
    Single-turn RAG chain using LangChain LCEL.

    Pipeline:
      question
        → retriever (FAISS MMR)  → _format_docs  ┐
        → RunnablePassthrough()                   ┤ → prompt → llm → str
    """

    def __init__(self, vsm: VectorStoreManager, streaming=False):
        self.vsm  = vsm
        self.llm  = get_llm(streaming=streaming)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", SYSTEM_PROMPT),
            ("human", "{question}"),
        ])

        retriever = vsm.as_retriever()

        # ── The LCEL pipeline ──────────────────────────────────────
        # Input: question string
        # ↓  branch 1: question → retriever → _format_docs → "context"
        # ↓  branch 2: question → passthrough → "question"
        # ↓  both merge into {context, question} dict
        # ↓  prompt fills template with the dict
        # ↓  llm generates the answer
        # ↓  StrOutputParser extracts the text string
        self.chain = (
            {
                "context":  retriever | _format_docs,
                "question": RunnablePassthrough(),
            }
            | self.prompt
            | self.llm
            | StrOutputParser()
        )
        print("✅ RAGChain ready")

    def invoke(self, question: str) -> str:
        """Ask a question. Returns the full answer string."""
        return self.chain.invoke(question)

    def invoke_with_sources(self, question: str) -> dict:
        """Returns {answer, sources} where sources is a list of cited chunks."""
        source_docs = self.vsm.as_retriever().invoke(question)
        answer      = self.invoke(question)
        return {
            "answer":  answer,
            "sources": [
                {
                    "title":     d.metadata.get("title", ""),
                    "authors":   d.metadata.get("authors", ""),
                    "published": d.metadata.get("published", ""),
                    "url":       d.metadata.get("url", ""),
                    "snippet":   d.page_content[:300],
                }
                for d in source_docs
            ],
        }

    def stream(self, question: str):
        """Stream the answer token by token."""
        yield from self.chain.stream(question)


rag = RAGChain(vsm)
print("RAGChain is built on top of VectorStoreManager (vsm from Step 5)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Ask a single-turn question with sources
# ─────────────────────────────────────────────────────────────────

question = "What is the ReAct framework and how does it combine reasoning and acting?"

print(f"Question: {question}")
print("Retrieving + generating...")
print()

result = rag.invoke_with_sources(question)

print("=" * 60)
print("ANSWER:")
print("=" * 60)
print(result["answer"])

print()
print("=" * 60)
print("SOURCES USED:")
print("=" * 60)
for i, s in enumerate(result["sources"], 1):
    print(f"[{i}] {s['title']}")
    print(f"    {s['authors']} ({s['published']})")
    print(f"    {s['url']}")
    print(f"    Snippet: {s['snippet'][:150]}...")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Streaming output (token by token)
# ─────────────────────────────────────────────────────────────────
# In the Streamlit app, this is used with st.write_stream()
# Here we just print tokens as they arrive.

print("Streaming answer:")
print("-" * 40)

for token in rag.stream("What are the main contributions of the papers in the knowledge base?"):
    print(token, end="", flush=True)

print()
print("-" * 40)
print("(Streaming complete)")

---
## Step 7 — `rag_chain.py` — Conversational RAG (Multi-turn)

**What this does:**  
Extends single-turn RAG to support follow-up questions that reference previous 
answers. The key addition is a **condense step** that rewrites vague follow-ups 
into self-contained questions before retrieval.

**Why this is needed:**  
If you ask *"How does it compare to CoT?"* after asking about ReAct, the retriever 
has no idea what "it" refers to. The condense step rewrites this to  
*"How does the ReAct framework compare to Chain-of-Thought (CoT) prompting?"*  
before searching FAISS.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# rag_chain.py — ConversationalRAGChain
# ─────────────────────────────────────────────────────────────────

from langchain_core.prompts import MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

class ConversationalRAGChain:
    """
    Multi-turn RAG with chat history.

    Two-step process per turn:
    1. Condense:  history + follow-up → standalone question
    2. Answer:    standalone question → retrieve → LLM → answer

    Sessions are keyed by session_id string.
    """

    def __init__(self, vsm: VectorStoreManager):
        self.vsm      = vsm
        self.llm      = get_llm()
        self._sessions = {}  # session_id → ChatMessageHistory

        retriever = vsm.as_retriever()

        # ── Step 1: condense follow-up to standalone question ──────
        condense_prompt = ChatPromptTemplate.from_messages([
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}"),
            ("human",
             "Given the conversation above, rewrite my follow-up as a "
             "standalone question with all necessary context."),
        ])
        self._condense = condense_prompt | self.llm | StrOutputParser()

        # ── Step 2: RAG answer with history in prompt ──────────────
        answer_prompt = ChatPromptTemplate.from_messages([
            ("system", SYSTEM_PROMPT),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}"),
        ])

        def retrieve_and_format(inputs: dict) -> dict:
            q    = inputs.get("standalone_question", inputs["question"])
            docs = retriever.invoke(q)
            return {
                "context":      _format_docs(docs),
                "question":     inputs["question"],
                "chat_history": inputs.get("chat_history", []),
            }

        self._answer_chain = (
            RunnablePassthrough.assign(standalone_question=self._condense)
            | retrieve_and_format
            | answer_prompt
            | self.llm
            | StrOutputParser()
        )
        print("✅ ConversationalRAGChain ready")

    def chat(self, question: str, session_id="default") -> str:
        """Send a message. History is maintained automatically per session_id."""
        history = self._sessions.setdefault(session_id, ChatMessageHistory())
        answer  = self._answer_chain.invoke({
            "question":     question,
            "chat_history": history.messages,
        })
        history.add_user_message(question)
        history.add_ai_message(answer)
        return answer

    def get_history(self, session_id="default"):
        history = self._sessions.get(session_id)
        if not history:
            return []
        return [
            {"role": msg.type, "content": msg.content}
            for msg in history.messages
        ]

    def clear_history(self, session_id="default"):
        if session_id in self._sessions:
            self._sessions[session_id].clear()
            print(f"History cleared for session '{session_id}'")


conv = ConversationalRAGChain(vsm)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Multi-turn conversation
# ─────────────────────────────────────────────────────────────────
# Notice turn 2 is a vague follow-up — "How does it differ...?"
# The condense step internally rewrites it before retrieval.

SESSION = "notebook-demo"

# ── Turn 1 ────────────────────────────────────────────────────────
q1 = "What is the main idea behind agentic AI systems?"
print(f"👤 Turn 1: {q1}")
print()
a1 = conv.chat(q1, session_id=SESSION)
print(f"🤖 Answer:\n{a1}")

print()
print("-" * 60)
print()

# ── Turn 2: vague follow-up ───────────────────────────────────────
q2 = "How does it differ from traditional AI pipelines?"
print(f"👤 Turn 2: {q2}")
print("    (Note: 'it' is ambiguous — condense step resolves this)")
print()
a2 = conv.chat(q2, session_id=SESSION)
print(f"🤖 Answer:\n{a2}")

# ── Show history ──────────────────────────────────────────────────
print()
print("=" * 60)
print("Chat history:")
print("=" * 60)
for msg in conv.get_history(SESSION):
    role = "👤" if msg["role"] == "human" else "🤖"
    print(f"{role} {msg['content'][:100]}...")
    print()

---
## Step 8 — `research_agent.py` — The Autonomous ReAct Agent

**What this does:**  
Wraps the entire system (fetcher + processor + vector store + RAG chain) into  
a LangChain ReAct agent. The agent autonomously decides which tool to call,  
observes the result, reasons about what to do next, and repeats until done.

**The 4 tools:**
| Tool | When used | Calls internally |
|---|---|---|
| `search_arxiv` | First, always | `ArxivFetcher.search()` |
| `ingest_papers` | After search, before query | `download_pdfs()` + `process_papers()` + `vsm.add_documents()` |
| `query_knowledge_base` | Main Q&A tool | `RAGChain.invoke_with_sources()` |
| `summarize_paper` | For quick abstract summaries | `get_llm()` directly |

**ReAct loop:**  
```
Thought → Action → Observation → Thought → Action → Observation → ... → Final Answer
```

In [ ]:
# ─────────────────────────────────────────────────────────────────
# research_agent.py — full implementation
# ─────────────────────────────────────────────────────────────────

from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import StructuredTool
from pydantic import BaseModel as PydanticModel, Field

# ── Shared state across all tools ─────────────────────────────────
class AgentState:
    """Mutable state shared by all tools in one agent session."""
    def __init__(self):
        self.fetcher       = ArxivFetcher()
        self.processor     = DocumentProcessor()
        self.vsm           = VectorStoreManager()
        self.rag_chain     = None
        self.ingested_ids  = set()
        self._paper_cache  = {}
        self._store_ready  = False

    def ensure_rag_chain(self):
        if self.rag_chain is None:
            if not self._store_ready:
                raise RuntimeError("Ingest papers first before querying.")
            self.rag_chain = RAGChain(self.vsm)
        return self.rag_chain


# ── Input schemas (Pydantic) ──────────────────────────────────────
class SearchInput(PydanticModel):
    query:       str            = Field(description="arXiv search query")
    max_results: int            = Field(default=4)
    categories:  Optional[List[str]] = Field(default=None)

class IngestInput(PydanticModel):
    paper_ids: List[str] = Field(description="List of arXiv paper IDs to ingest")

class QueryInput(PydanticModel):
    question: str = Field(description="Question to answer from knowledge base")

class SummarizeInput(PydanticModel):
    paper_id: str = Field(description="arXiv paper ID to summarize")


# ── Tool factory ──────────────────────────────────────────────────
def build_tools(state: AgentState):

    def search_arxiv(query, max_results=4, categories=None):
        ps = state.fetcher.search(query, max_results=max_results, categories=categories)
        if not ps:
            return "No papers found."
        state._paper_cache = {p.paper_id: p for p in ps}
        lines = [f"Found {len(ps)} papers:\n"]
        for p in ps:
            lines.append(f"  ID: {p.paper_id} | {p.title} ({p.published})")
            lines.append(f"  Abstract: {p.abstract[:180]}...\n")
        return "\n".join(lines)

    def ingest_papers(paper_ids):
        to_ingest = []
        for pid in paper_ids:
            if pid in state.ingested_ids:
                continue
            paper = state._paper_cache.get(pid)
            if not paper:
                fetched = state.fetcher.fetch_by_ids([pid])
                if fetched:
                    paper = fetched[0]
                else:
                    return f"Paper {pid} not found."
            to_ingest.append(paper)

        if not to_ingest:
            return "All papers already in knowledge base."

        to_ingest   = state.fetcher.download_pdfs(to_ingest)
        docs        = state.processor.process_papers(to_ingest)
        if not docs:
            return "No content extracted."

        try:
            state.vsm.load_or_create(docs)
        except Exception:
            state.vsm.add_documents(docs)

        state._store_ready = True
        state.rag_chain    = None  # reset so it rebuilds with new docs
        for p in to_ingest:
            state.ingested_ids.add(p.paper_id)

        return (f"Ingested {len(to_ingest)} paper(s), {len(docs)} chunks. "
                f"Knowledge base ready for queries.")

    def query_knowledge_base(question):
        rc     = state.ensure_rag_chain()
        result = rc.invoke_with_sources(question)
        src    = "\n".join(
            f"  • {s['title']} ({s['published']}) — {s['url']}"
            for s in result["sources"][:3]
        )
        return result["answer"] + f"\n\nSources:\n{src}"

    def summarize_paper(paper_id):
        paper = state._paper_cache.get(paper_id)
        if not paper:
            fetched = state.fetcher.fetch_by_ids([paper_id])
            if not fetched:
                return f"Paper {paper_id} not found."
            paper = fetched[0]

        from langchain_core.messages import HumanMessage
        prompt = (
            f"Summarize in 5 bullet points (problem, method, contributions, "
            f"results, limitations):\n\nTitle: {paper.title}\n"
            f"Authors: {', '.join(paper.authors[:3])}\n"
            f"Abstract:\n{paper.abstract}"
        )
        resp = get_llm().invoke([HumanMessage(content=prompt)])
        return f"Summary of '{paper.title}':\n\n{resp.content}"

    return [
        StructuredTool(name="search_arxiv", func=search_arxiv,
            args_schema=SearchInput,
            description="Search arXiv for papers. Use this FIRST."),
        StructuredTool(name="ingest_papers", func=ingest_papers,
            args_schema=IngestInput,
            description="Download and index paper IDs into the knowledge base."),
        StructuredTool(name="query_knowledge_base", func=query_knowledge_base,
            args_schema=QueryInput,
            description="Answer a question using the ingested papers (RAG). Requires ingest first."),
        StructuredTool(name="summarize_paper", func=summarize_paper,
            args_schema=SummarizeInput,
            description="Bullet-point summary of one paper by its arXiv ID."),
    ]


# ── ReAct prompt ──────────────────────────────────────────────────
REACT_PROMPT = PromptTemplate.from_template("""\
You are an expert Agentic AI research assistant.
You have access to these tools:
{tools}

Use this EXACT format:
Question: the input question
Thought: what to do next
Action: tool name (one of [{tool_names}])
Action Input: JSON input for the tool
Observation: result of the action
... (repeat as needed)
Thought: I now know the final answer
Final Answer: complete answer

Strategy: search_arxiv first → ingest_papers → query_knowledge_base

Question: {input}
Thought: {agent_scratchpad}""")


# ── ResearchAgent ─────────────────────────────────────────────────
class ResearchAgent:
    """Autonomous agent that searches, ingests, and answers questions."""

    def __init__(self):
        self.state   = AgentState()
        self.tools   = build_tools(self.state)
        agent        = create_react_agent(
            llm=get_llm(), tools=self.tools, prompt=REACT_PROMPT
        )
        self.executor = AgentExecutor(
            agent=agent, tools=self.tools,
            verbose=True,
            max_iterations=settings.AGENT_MAX_ITERATIONS,
            handle_parsing_errors=True,
            return_intermediate_steps=True,
        )
        print("✅ ResearchAgent ready")

    def run(self, query: str) -> dict:
        """Run the agent. Returns {output, steps}."""
        result = self.executor.invoke({"input": query})
        return {
            "output": result.get("output", ""),
            "steps":  [
                {
                    "action":       step[0].tool,
                    "action_input": step[0].tool_input,
                    "observation":  str(step[1])[:400],
                }
                for step in result.get("intermediate_steps", [])
            ],
        }

print("✅ ResearchAgent class defined")

---
## Step 9 — Full End-to-End Agent Run

This is the complete system in action. The agent will:
1. Receive your research question
2. Decide to call `search_arxiv` with relevant keywords
3. Select the most relevant papers and call `ingest_papers`
4. Call `query_knowledge_base` to generate a RAG-grounded answer
5. Return the final synthesized answer with citations

Watch the `Thought → Action → Observation` loop printed below — that's the ReAct reasoning trace.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUN: Full autonomous agent
# ─────────────────────────────────────────────────────────────────
# This is the complete end-to-end pipeline.
# verbose=True prints the full ReAct reasoning trace.
# Expect this to take 1-3 minutes (search + download + embed + generate).

agent = ResearchAgent()

print()
print("=" * 60)
print("Running agent on research question...")
print("=" * 60)
print()

QUERY = "What are the key differences between ReAct and Chain-of-Thought prompting for LLM agents?"
print(f"Question: {QUERY}")
print()

result = agent.run(QUERY)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Display the structured result
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["output"])

print()
print("=" * 60)
print(f"AGENT REASONING STEPS ({len(result['steps'])} steps)")
print("=" * 60)
for i, step in enumerate(result["steps"], 1):
    print(f"\nStep {i}: {step['action']}")
    print(f"  Input:       {str(step['action_input'])[:120]}...")
    print(f"  Observation: {step['observation'][:200]}...")

---
## Step 10 — Quick Tool Tests (run each tool independently)

You can also call each tool directly without going through the full agent.  
Useful for debugging or building custom pipelines.

In [ ]:
# Tool 1: search_arxiv directly
fetcher2 = ArxivFetcher(max_results=3)
papers2  = fetcher2.search("memory LLM agents long-term", categories=["cs.AI"])

print(f"\nFound {len(papers2)} papers in cs.AI:")
for p in papers2:
    print(f"  {p.paper_id}: {p.title[:70]}")

In [ ]:
# Tool 4: summarize_paper directly (no ingestion needed)
# Uses the first paper we fetched earlier

if papers:
    paper_to_summarize = papers[0]
    print(f"Summarizing: {paper_to_summarize.title}")
    print()

    from langchain_core.messages import HumanMessage
    prompt = (
        f"Summarize in 5 bullet points covering: problem, method, "
        f"contributions, results, limitations.\n\n"
        f"Title: {paper_to_summarize.title}\n"
        f"Authors: {', '.join(paper_to_summarize.authors[:3])}\n"
        f"Abstract:\n{paper_to_summarize.abstract}"
    )
    response = get_llm().invoke([HumanMessage(content=prompt)])
    print(response.content)

In [ ]:
# Compare MMR vs plain similarity retrieval
query_test = "how do agents use tools and external APIs"

print("=" * 60)
print(f"Query: '{query_test}'")
print("=" * 60)

# MMR — diverse results
mmr_retriever = vsm.as_retriever(search_type="mmr", k=3)
mmr_results   = mmr_retriever.invoke(query_test)
print("\nMMR results (diverse):")
for r in mmr_results:
    print(f"  [{r.metadata['paper_id']}] chunk {r.metadata['chunk_index']} | {r.page_content[:80]}...")

# Similarity — may have duplicates
sim_retriever = vsm.as_retriever(search_type="similarity", k=3)
sim_results   = sim_retriever.invoke(query_test)
print("\nSimilarity results (may repeat):")
for r in sim_results:
    print(f"  [{r.metadata['paper_id']}] chunk {r.metadata['chunk_index']} | {r.page_content[:80]}...")

---
## Summary — what you built

```
config.py
  └── loads .env → typed Settings object
        ↓ used by every module below

arxiv_fetcher.py
  └── ArxivFetcher
        ├── search(query)         → List[ArxivPaper]  (metadata only)
        ├── download_pdfs(papers) → List[ArxivPaper]  (+ local PDF paths)
        └── fetch_by_ids(ids)     → List[ArxivPaper]

document_processor.py
  └── DocumentProcessor
        └── process_papers(papers) → List[Document]   (chunks + metadata)

vector_store.py
  └── VectorStoreManager
        ├── load_or_create(docs)   → builds/loads FAISS index
        ├── similarity_search(q)   → List[Document]
        └── as_retriever()         → LangChain Retriever

rag_chain.py
  ├── RAGChain
  │     ├── invoke(q)              → answer string
  │     ├── invoke_with_sources(q) → {answer, sources}
  │     └── stream(q)              → token iterator
  └── ConversationalRAGChain
        └── chat(q, session_id)    → answer (history-aware)

research_agent.py
  └── ResearchAgent
        └── run(query)             → {output, steps}
              ↑ uses all 4 tools autonomously

app.py  (Streamlit — run with: streamlit run app.py)
  ├── Agent mode   → ResearchAgent.run()
  ├── RAG Chat     → ConversationalRAGChain.chat()
  └── Search       → ArxivFetcher + DocumentProcessor + VectorStoreManager
```

**To launch the full app:**
```bash
streamlit run app.py
```